# Spinor band structure from a magnetic texture

This script:
1. Loads **collinear** Wannier90 TB models (spin-up and spin-down) using PythTB/Wannier90.
2. Builds a **spinful** (spinor) TB model with diagonal spin blocks.
3. Assigns a local **unitary SU(2) rotation** $u_i$ to each Wannier orbital center from a prescribed texture $\mathbf n(\mathbf r)$.
4. Rotates onsite and hopping using the following conventions:

$$
\varepsilon'_i = u_i\,\varepsilon_i\,u_i^\dagger,\qquad
t'_{ij} = u_i\, t_{ij}\, u_j^\dagger.
$$

We then diagonalize the rotated spinor TB model along a k-path and plot $E-E_F$.

---
## Texture and SU(2) mapping
We use the analytic “skyrmion-like” unit vector field:

$$
m_x=\frac{2x}{x^2+y^2+r_0^2},\quad
m_y=\frac{2y}{x^2+y^2+r_0^2},\quad
m_z=\frac{x^2+y^2-r_0^2}{x^2+y^2+r_0^2},
$$

giving $\mathbf n(\mathbf r)=(m_x,m_y,m_z)$ (normalized numerically).

Given a reference spin axis $\mathbf s_{\rm ref}$ (typically $\hat z$), we construct an SU(2) matrix
$u(\mathbf r)$ such that in the spinor representation the local spin quantization direction aligns with $\mathbf n(\mathbf r)$.

---
## Notes / prerequisites
- Ensure `tb_spinor.py` and `magn_spinor.py` are in the same directory as this script.
- If you’re using `tb_spinor.py` exactly as pasted earlier, check that `raise valueerror(...)` is corrected to `raise ValueError(...)` (otherwise it will crash).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pythtb import w90

from tb_spinor import *
#from tb_spinor_wigner import *
from magn_spinor import *

import plotly.graph_objects as go

In [ ]:
# USER SETTINGS (collinear TB + plotting)
seed_up = "wannier90.1"
seed_dn = "wannier90.2"

w90_dir = "/Users/guymoore/Documents/ResearchProjects/magnetism/BiFeO3_TB/00_R3c_cl"

n_k_per_segment = 32
fermi_level = 4.50820836  # set to EF (eV) if you know it

# k-path settings (fractional reciprocal coords)
G = np.array([0.0, 0.0, 0.0])
k_delta = 0.1
S1 = np.array([-k_delta, 0.5, k_delta], dtype=float)
S2 = np.array([k_delta, 0.5, -k_delta], dtype=float)

k_nodes = [
    ("S1", S1),
    ("Γ", G),
    ("S2", S2),
]

# USER SETTINGS (texture + SU(2) assignment)
# Cone-plot visualization window (only for plotting the texture)
l_x = 1.0
n_x = 3
texture_z_plane = 0.0  # used only for the cone plot

# Texture parameters in the analytic formula
r0 = 1.0

# When assigning u(r) to Wannier orbital centers:
# the skyrmion texture uses x,y in the *same units* as your orbital Cartesian coordinates.
# If your orbital centers are in Angstrom, leave texture_scale=1.0; if not, tune it.
texture_scale = 1.0
texture_center_xy = (0.0, 0.0)  # (x_center, y_center) shift in Cartesian coordinates

# Reference spin direction for SU(2) rotations
# Our u(r) construction aligns the reference axis to n(r).
sref = np.array([0.0, 0.0, 1.0], dtype=float)

# Supercell toggle:
# - None => rotate only within the reference cell (cell_key = (0,0,0))
# - not None => you must supply u for each (supercell internal cell, wf_index)

# sc_red_lat = None
# sc_red_lat = np.diag([1, 1, 1])
sc_red_lat = np.diag([n_x, n_x, 1])

to_home = True

# pruning hoppings when importing Wannier90 into PythTB
min_hopping_norm = 1.0e-2
# --------------------------
# MAGNETIC TEXTURE / INITIALIZATION SELECTION
# --------------------------

mag_fn = magnetization_texture_skyrmion_00

# Option 2: Uniform Collinear Magnetization along custom 3D vector vec=[mx, my, mz]

# mag_fn = lambda x, y, z=0.0, r0=1.0: magnetization_collinear(x, y, z=z, r0=r0, vec=[1.0, 0.0, 0.0])  # +x
# mag_fn = lambda x, y, z=0.0, r0=1.0: magnetization_collinear(x, y, z=z, r0=r0, vec=[0.0, 1.0, 0.0])  # +y
# mag_fn = lambda x, y, z=0.0, r0=1.0: magnetization_collinear(x, y, z=z, r0=r0, vec=[0.0, 0.0, 1.0])  # +z


In [ ]:
# --------------------------
# Helper: build a k-path (same style as your collinear script)
# --------------------------
def build_kpath(k_nodes, n_per_segment):
    labels = [k[0] for k in k_nodes]
    kpts = np.array([k[1] for k in k_nodes], dtype=float)

    k_list = []
    x_list = []
    tick_positions = [0.0]
    x = 0.0

    for i in range(len(kpts) - 1):
        k0 = kpts[i]
        k1 = kpts[i + 1]
        for j in range(n_per_segment):
            t = j / float(n_per_segment)
            k = (1 - t) * k0 + t * k1
            k_list.append(k)
            x_list.append(x)
            if j < n_per_segment - 1:
                dk = np.linalg.norm((k1 - k0) / n_per_segment)
                x += dk
        tick_positions.append(x)

    k_list.append(kpts[-1])
    x_list.append(x)

    return np.array(k_list), np.array(x_list), labels, np.array(tick_positions)

In [ ]:
k_vec, x_vec, labels, tick_pos = build_kpath(k_nodes, n_k_per_segment)

## Plot the magnetic texture (cone plot)
This is the same cone plot you had earlier, showing the unit vector field $\mathbf n(\mathbf r)$ over an $x$-$y$ grid.

In [ ]:
xs = np.linspace(-0.5 * l_x, 0.5 * l_x, n_x, endpoint=True)
ys = np.linspace(-0.5 * l_x, 0.5 * l_x, n_x, endpoint=True)

xg, yg = np.meshgrid(xs, ys, indexing="xy")
zg = texture_z_plane * np.ones_like(xg)

n_grid = mag_fn(xg, yg, z=texture_z_plane, r0=r0)
mx, my, mz = n_grid[..., 0], n_grid[..., 1], n_grid[..., 2]

fig = go.Figure(
    data=go.Cone(
        x=xg.ravel(),
        y=yg.ravel(),
        z=zg.ravel(),
        u=mx.ravel(),
        v=my.ravel(),
        w=mz.ravel(),
        colorscale="Viridis",
        showscale=False,
        sizemode="scaled",
        sizeref=4.0,
        anchor="tail",
    )
)

# rough aspect (purely for visualization)
xl, yl, zl = 1.0, 1.0, 1.0 / max(n_x, 1)
fig.update_layout(
    width=800,
    height=600,
    margin={"l": 0, "r": 0, "t": 0, "b": 0},
    scene=dict(
        aspectratio=dict(x=xl, y=yl, z=zl),
        camera_eye=dict(x=1.0, y=1.0, z=1.0),
        zaxis=dict(visible=False),
    ),
)
fig.show()


## Load Wannier90 TB models and build the spinful model
We use the collinear Wannier90 outputs:
- spin-up Wannier set $\Rightarrow$ `tb_up` (nspin=1)
- spin-down Wannier set $\Rightarrow$ `tb_dn` (nspin=1)

Then `build_spinful_from_collinear()` creates an nspin=2 spinor model with diagonal spin blocks.

In [ ]:
# --------------------------
# Load Wannier90 + build PythTB models
# --------------------------
w90_up = w90(w90_dir, seed_up)
w90_dn = w90(w90_dir, seed_dn)

print("Creating TB model...")

tb_up = w90_up.model(min_hopping_norm=min_hopping_norm)
tb_dn = w90_dn.model(min_hopping_norm=min_hopping_norm)

print("Solving TB model... (texture rotation happens next)")

In [ ]:
orb_up = tb_up.get_orb()
orb_dn = tb_dn.get_orb()

diff_raw = orb_up - orb_dn
diff_wrapped = diff_raw - np.round(diff_raw)  # wrap into ~[-0.5,0.5]

print("max raw  |diff|      =", np.max(np.abs(diff_raw)))
print("max wrapped |diff|  =", np.max(np.abs(diff_wrapped)))

In [ ]:
tb_spinful = build_spinful_from_collinear_intersection(tb_up, tb_dn, fermi_level=fermi_level)

dim_r = tb_spinful._dim_r
base_norb = tb_spinful._norb

print(f"Spinful model: dim_r={dim_r}, base_norb={base_norb}, nspin={tb_spinful._nspin}")

## Construct $u$-matrices on every Wannier orbital center
We build an SU(2) unitary rotation for each Wannier orbital at Cartesian position
$\mathbf r = \mathbf r_{\text{orb}}^{\text{(reduced)}} \cdot \text{lat}$.

Then we evaluate the texture $\mathbf n(\mathbf r)$ and compute
$u(\mathbf r)\equiv u_i$.

Your `tb_spinor.py` expects:

- `u_samples[(cell_r_int_tuple, wf_index)] = 2x2 complex unitary`

If `sc_red_lat is None`, we only use the reference cell with `cell_r_int_tuple=(0,0,0)`.

In [ ]:
# --------------------------
# Build the supercell geometry if needed
# --------------------------
if sc_red_lat is not None:
    sc_tb, sc_vectors = tb_spinful.make_supercell(
        sc_red_lat, return_sc_vectors=True, to_home=to_home
    )
else:
    sc_tb = tb_spinful
    sc_vectors = [np.zeros(dim_r, dtype=int)]

lat_sc = sc_tb.get_lat()
orb_sc = sc_tb.get_orb()  # reduced coords for each orbital in the sc model

# sanity check
num_sc = len(sc_vectors)
if sc_tb._norb != base_norb * num_sc:
    raise ValueError(
        f"Unexpected supercell orbital count: sc_tb._norb={sc_tb._norb}, "
        f"expected base_norb*num_sc={base_norb*num_sc}"
    )

u_samples = {}
x_center, y_center = texture_center_xy

for sc_i, cell_r in enumerate(sc_vectors):
    cell_key = tuple(int(x) for x in np.asarray(cell_r, dtype=int))

    for wf_i in range(base_norb):
        orb_i = sc_i * base_norb + wf_i

        # Cartesian position of Wannier orbital center:
        # orb_sc[orb_i] are reduced coordinates in dim_r; lat_sc maps to Cartesian.
        r_cart = np.dot(orb_sc[orb_i], lat_sc)  # (dim_r,)

        x_cart = float(r_cart[0])
        y_cart = float(r_cart[1])
        z_cart = float(r_cart[2]) if dim_r >= 3 else 0.0

        # shift & scale before evaluating texture
        x_tex = texture_scale * (x_cart - x_center)
        y_tex = texture_scale * (y_cart - y_center)

        n_loc = np.squeeze(mag_fn(x_tex, y_tex, z=z_cart, r0=r0))

        u_loc = su2_from_ref_to_n(sref=sref, n=n_loc)  # (2,2)

        u_samples[(cell_key, int(wf_i))] = u_loc

print(f"Built u_samples with {len(u_samples)} entries.")

# Optional: SU(2) sign continuity gauge fix
# (Does not affect eigenvalues; can help if you later use eigenvectors for Berry phases.)
try:
    ordered_keys = sorted(u_samples.keys())
    u_stack = np.array([u_samples[k] for k in ordered_keys], dtype=complex)
    u_stack_fixed = su2_fix_sign_continuity(u_stack)
    for k, u in zip(ordered_keys, u_stack_fixed):
        u_samples[k] = u
        # print(u)
    print("Applied SU(2) sign continuity fix.")
except Exception as e:
    print(f"Skipping sign continuity fix due to: {e}")

## Compute rotated spinor bands
We call:

```python
compute_rotated_bands_from_cellwf_samples(...)
```

which internally:
- rebuilds the spinful model from `tb_up`, `tb_dn`
- applies the local rotations to onsite and hopping
- diagonalizes along `k_vec`.

In [ ]:
# E_rot = compute_rotated_bands_from_cellwf_samples_numba(
#     tb_up, tb_dn, k_vec, u_samples,
#     sc_red_lat=sc_red_lat,
#     numba_threads=8,
#     limit_blas_threads=True,
#     blas_threads=1,
# )

# # # with eigenvectors:
# # E_rot, Evecs_rot = compute_rotated_bands_from_cellwf_samples_numba(
# #     tb_up, tb_dn, k_vec, u_samples,
# #     sc_red_lat=sc_red_lat,
# #     eig_vectors=True,
# #     numba_threads=8,
# # )

In [ ]:
# Convert primitive k-path to supercell reciprocal units:
k_sc_input = k_vec * np.diag(sc_red_lat)

# eigenvalues + eigenvectors
E_rot, Evecs_rot = compute_rotated_bands_from_cellwf_samples_alt(
    tb_up, tb_dn, k_sc_input, u_samples,
    sc_red_lat=sc_red_lat,
    eig_vectors=True,
    eig_backend="parallel",
)


In [ ]:
# E_rot = compute_rotated_bands_from_cellwf_samples(
#     tb_up=tb_up,
#     tb_dn=tb_dn,
#     k_vec=k_vec,
#     u_samples=u_samples,
#     sc_red_lat=sc_red_lat,
#     to_home=to_home,
#     # fermi_level=fermi_level,
#     fermi_level=0.0,
# )

In [ ]:
print("Bands computed.")
print("E_rot shape:", E_rot.shape)

In [ ]:
plt.figure(dpi=300, figsize=(4.5, 4.5))

for n in range(E_rot.shape[0]):
    plt.plot(
        x_vec,
        E_rot[n, :]-fermi_level,
        linewidth=0.9,
        alpha=0.9,
        color="k",
        linestyle="-",
    )

for xp in tick_pos:
    plt.axvline(xp, linewidth=0.8, color="gray", alpha=0.7)

plt.xticks(tick_pos, labels)
plt.ylabel(r"Energy $E - E_F$ (eV)")
plt.xlabel("k-path")

if sc_red_lat is None:
    plt.title("Spinor TB bands (texture): primitive-cell assignment")
else:
    plt.title(f"Spinor TB bands (texture) \n supercell: {sc_red_lat.tolist()}")

plt.tight_layout()
plt.show()

## Compute DOS

In [ ]:
# evals: (nb, nk), evecs: (nb, nk, nbasis)  [or (nk, nbasis, nb)]
dos_out = compute_total_and_projected_dos(
    evals=E_rot,
    evecs=Evecs_rot,
    kpoints=k_vec,
    projector={"indices": [0,1,2,3,4,5,6,7,8,9]},   # project onto selected Wannier basis indices
    sigma=0.02,
    n_energy=3000,
    return_components=True,
)

energy = -fermi_level+dos_out["energy"]
dos = dos_out["dos"]
pdos = dos_out["pdos"]

In [ ]:
plt.figure(figsize=(6,4), dpi=300)
plt.plot(energy, dos, lw=2, label="total dos", color="black")
# plt.plot(energy, pdos, lw=2, label="projected dos", color="tab:red")
plt.xlabel("Energy (ev)")
plt.ylabel("DOS (states / ev)")
# plt.title("total and projected density of states")
plt.legend(frameon=False)
plt.tight_layout()
plt.show()

## Perform unfolding

In [ ]:
def gaussian_spectral_from_bands(
    E,              # (nb, nk)
    W,              # (nb, nk)  scalar spectral weight
    S=None,         # (nb, nk, 3) spin weights; optional
    e_grid=None,
    e_min=None,
    e_max=None,
    nE=600,
    sigma=0.03
):
    E = np.asarray(E, float)
    W = np.asarray(W, float)
    nb, nk = E.shape

    if e_grid is None:
        if e_min is None: e_min = E.min() - 5*sigma
        if e_max is None: e_max = E.max() + 5*sigma
        e_grid = np.linspace(e_min, e_max, nE)
    else:
        e_grid = np.asarray(e_grid, float)
        nE = e_grid.size

    A = np.zeros((nk, nE), float)
    Axyz = np.zeros((nk, nE, 3), float) if S is not None else None

    if S is not None:
        S_abs = np.abs(np.asarray(S, float))

    norm = 1.0/(np.sqrt(2*np.pi)*sigma)

    for ik in range(nk):
        for jb in range(nb):
            g = norm*np.exp(-0.5*((e_grid - E[jb,ik])/sigma)**2)  # (nE,)
            A[ik] += W[jb,ik]*g
            if S is not None:
                Axyz[ik,:,0] += S_abs[jb,ik,0]*g
                Axyz[ik,:,1] += S_abs[jb,ik,1]*g
                Axyz[ik,:,2] += S_abs[jb,ik,2]*g

    return e_grid, A, Axyz


In [ ]:
from unfold_spin_project import *

proj = project_sc_bands_on_reference(
    tb_up=tb_up,
    tb_dn=tb_dn,
    E_sc=E_rot,
    V_sc=Evecs_rot,
    k_s_list=k_sc_input,
    sc_red_lat=sc_red_lat,
    fermi_level=fermi_level,
    nb_ref_keep=2*tb_up._norb
)

# Momentum-filtering: pick the unique matching sector iq for each k-point ik along the path:
nband_sc, nk = E_rot.shape
W_pick = np.zeros((nband_sc, nk))
S_pick = np.zeros((nband_sc, nk, 3))

for ik in range(nk):
    k_p_prim = proj["k_p"][ik] / np.diag(sc_red_lat)
    diffs = np.linalg.norm(k_p_prim - k_vec[ik], axis=-1)
    iq_match = np.argmin(diffs)
    W_pick[:, ik] = proj["weights"][:, ik, iq_match, :].sum(axis=-1)
    S_pick[:, ik, :] = proj["spin_xyz"][:, ik, iq_match, :, :].sum(axis=-2)

W_iq = W_pick
S_iq = S_pick


In [ ]:
# Example: picked per-(band,k) values
# W_pick: (nb,nk), S_pick: (nb,nk,3), E_rot: (nb,nk)
e_grid, A, Axyz = gaussian_spectral_from_bands(
    E=E_rot - fermi_level,
    W=W_iq,
    S=S_iq,
    sigma=0.03,
    nE=700
)

In [ ]:
eps = 1e-12
rgb = np.abs(Axyz)                              # (nk,nE,3)
rgb /= np.maximum(np.sum(rgb, axis=2, keepdims=True), eps)  # normalize color
intensity = A / (A.max() + eps)                 # 0..1
img = np.clip(rgb * intensity[...,None], 0, 1)  # (nk,nE,3)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(5,4), dpi=250)
# imshow expects (Ny,Nx,3): Ny=nE, Nx=nk
plt.imshow(
    np.transpose(img, (1,0,2)),
    origin="lower",
    aspect="auto",
    extent=[x_vec.min(), x_vec.max(), e_grid.min(), e_grid.max()]
)

for xp in tick_pos:
    plt.axvline(xp, color='w', lw=0.6, alpha=0.4)
plt.xticks(tick_pos, labels)
plt.ylabel("E - EF (eV)")
plt.xlabel("k-path")
plt.tight_layout()
plt.show()

In [ ]:
# plt.figure(dpi=250, figsize=(5,4))
# for j in range(E_rot.shape[0]):
#     c = RGB[j]  # (nk,3)
#     plt.scatter(x_vec, E_rot[j]-fermi_level, c=c, s=1, marker='.')
# for xp in tick_pos:
#     plt.axvline(xp, color='gray', lw=0.7)
# plt.xticks(tick_pos, labels)
# plt.ylabel("E-EF (eV)")
# plt.tight_layout()
# plt.show()

## Wigner diagonalization

In [ ]:
# # Notes:
# #  -  k_super:   single supercell momentum point (dim_k-compatible)
# #  -  k_p_list:  path in primitive/bz variable (nk_p, dim_r)

# E_wig, meta = compute_wigner_bands_for_fixed_k_and_r0(
#     tb_up,
#     tb_dn,
#     k_super=np.array([0,0,0]),
#     k_p_list=k_vec,
#     # k_super=k_vec,
#     # k_p_list=np.array([0,0,0]),
#     u_samples=u_samples,
#     sc_red_lat=sc_red_lat,
#     to_home=to_home,
#     fermi_level=fermi_level,
#     target_r0_cell_idx=0,
#     eig_vectors=False,
#     phase_2pi=True,
#     wigner_norm_mode="pairs",
# )

In [ ]:
# plt.figure(dpi=300, figsize=(4.5, 4.5))

# for n in range(E_wig.shape[0]):
#     plt.plot(
#         x_vec,
#         E_wig[n, :],
#         linewidth=0.9,
#         alpha=0.9,
#         color="k",
#         linestyle="-",
#     )

# for xp in tick_pos:
#     plt.axvline(xp, linewidth=0.8, color="gray", alpha=0.7)

# plt.xticks(tick_pos, labels)
# plt.ylabel(r"Energy $E - E_F$ (eV)")
# plt.xlabel("k-path")

# if sc_red_lat is None:
#     plt.title("Spinor TB bands (texture): primitive-cell assignment")
# else:
#     plt.title(f"Spinor TB bands (texture) \n supercell: {sc_red_lat.tolist()}")

# plt.tight_layout()
# plt.show()

## Spinor test

In [ ]:
# ---------- Pauli matrices ----------
I2 = np.eye(2, dtype=complex)
sx = np.array([[0, 1], [1, 0]], dtype=complex)
sy = np.array([[0, -1j], [1j, 0]], dtype=complex)
sz = np.array([[1, 0], [0, -1]], dtype=complex)
pauli = [sx, sy, sz]

rng = np.random.default_rng(7)

def make_t(eps, eta):
    """t = eps*I + eta·sigma"""
    return eps * I2 + eta[0] * sx + eta[1] * sy + eta[2] * sz

def reconstruct_eps_eta(t):
    """
    Reconstruct eps and eta from 2x2 matrix t using:
      eps = (1/2) Tr(t)
      eta_a = (1/2) Tr((t - eps I) sigma_a)
    """
    eps_rec = 0.5 * np.trace(t)

    t_minus = t - eps_rec * I2
    eta_rec = np.array(
        [0.5 * np.trace(t_minus @ s) for s in pauli],
        dtype=complex
    )
    return eps_rec, eta_rec

# ---------- test over several random "bonds" (i,j) ----------
n_test = 5
for n in range(n_test):
    # random complex eps and eta components
    eps = rng.normal() + 0*1j * rng.normal()
    eta = rng.normal(size=3) + 0*1j * rng.normal(size=3)

    t = make_t(eps, eta)
    eps_rec, eta_rec = reconstruct_eps_eta(t)

    print(f"\nTest {n}")
    print("eps true/rec :", eps, eps_rec)
    print("eta true     :", eta)
    print("eta rec      :", eta_rec)
    print("max|Δeps|    :", abs(eps - eps_rec))
    print("max|Δeta|    :", np.max(np.abs(eta - eta_rec)))

    # Optional strict check
    assert np.allclose(eps, eps_rec, atol=1e-12)
    assert np.allclose(eta, eta_rec, atol=1e-12)

print("\nAll tests passed.")

In [ ]:
n_loc = np.array([0.0, 0.0, 1.0])
u_loc = su2_from_ref_to_n(sref=sref, n=n_loc)  # should be identity
print(u_loc)

## Numerical Verification: Local-Frame Band Unfolding Benchmark

This section verifies that the local-frame spin-resolved unfolding scheme (`unfold_spin_project.py`) is 100% physically and numerically accurate.

**Benchmark Logic**:
- For a uniform supercell magnetization ($\mathbf{m}_i = (1,0,0)$ everywhere), the rotated supercell is physically equivalent to a zone-folded collinear crystal.
- When unfolded using local spin frames ($u_x$), the unfolded supercell band energies and local majority-spin weights $W$ must **agree 100% to machine precision** with the primitive collinear bands from `CollinearTB`.

In [ ]:
from unfold_spin_project import *

proj = project_sc_bands_on_reference(
    tb_up=tb_up,
    tb_dn=tb_dn,
    E_sc=E_rot,
    V_sc=Evecs_rot,
    k_s_list=k_sc_input,
    sc_red_lat=sc_red_lat,
    fermi_level=fermi_level,
    nb_ref_keep=2*tb_up._norb
)

# Momentum-filtering: pick the unique matching sector iq for each k-point ik along the path:
nband_sc, nk = E_rot.shape
W_pick = np.zeros((nband_sc, nk))
S_pick = np.zeros((nband_sc, nk, 3))

for ik in range(nk):
    k_p_prim = proj["k_p"][ik] / np.diag(sc_red_lat)
    diffs = np.linalg.norm(k_p_prim - k_vec[ik], axis=-1)
    iq_match = np.argmin(diffs)
    W_pick[:, ik] = proj["weights"][:, ik, iq_match, :].sum(axis=-1)
    S_pick[:, ik, :] = proj["spin_xyz"][:, ik, iq_match, :, :].sum(axis=-2)

W_iq = W_pick
S_iq = S_pick
